# 프롬프트 엔지니어링(Prompt Engineering)

프롬프트 엔지니어링은 단순히 질문을 던지는 것을 넘어, 모델의 작동 원리와 '인컨텍스트 러닝(In-context Learning)' 능력을 활용해 모델의 출력을 제어하는 프로세스이다. 이는 모델의 파라미터(가중치)를 직접 수정하지 않고도 모델의 성능을 특정 태스크에 맞게 조정하는 방법론이다.

**프롬프트의 핵심 구성 요소:**

효과적인 프롬프트는 일반적으로 다음의 4가지 요소를 포함한다.

* **지시문 (Instruction):** 모델이 수행해야 할 구체적인 작업(예: 요약하라, 분류하라, 번역하라 등).
* **문맥 (Context):** 모델이 작업을 더 잘 수행하도록 돕는 배경 정보나 제약 조건.
* **입력 데이터 (Input Data):** 처리가 필요한 실제 데이터.
* **출력 지시자 (Output Indicator):** 결과물의 형식이나 스타일 지정(예: 표로 정리하라, JSON 포맷으로 출력하라 등).

**프롬프트 엔지니어링의 중요성:**

* **성능 최적화:** 같은 모델이라도 프롬프트에 따라 성능 차이가 극심하다. 잘 설계된 프롬프트는 더 작은 모델로도 큰 모델 수준의 결과를 낼 수 있게 한다.
* **비용 효율성:** 불필요한 토큰 사용을 줄이고, 파인튜닝(Fine-tuning)에 비해 적은 비용으로 도메인 특화 작업을 수행할 수 있다.
* **한계 극복:** 모델의 환각 현상을 줄이고 최신 정보를 반영(RAG와 결합 시)하도록 유도할 수 있다.

In [2]:
from dotenv import load_dotenv # .env 환경변수 로드  
from openai import OpenAI      # 클라이언트 객체 생성 클래스
import os                      # 환경변수 접근
import json

load_dotenv()                  # .env파일 읽어와 환경변수 등록
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')  # .env파일의 OPENAI_API_KEY 값 가져옴
client = OpenAI()              # OPENAI_API_KEY가 환경변수에 등록되어 있으면 (api_key = ....) 생략 가능

In [ ]:
# Chat Completion API 호출
response = client.chat.completions.create(
    model = 'gpt-5.6-luna',
    messages = [
        # 시스템 프롬프트 : 모델의 페르소나 / 규칙 설정
        {
            "role" : "system",
            "content" : [
                {
                    "type" : "text",
                    # 모델의 역할 / 출력 예시 / 출력 규칙
                    "text" : "기자들이 송고한 제목에서 맞춤법/문법/의미/어조등을 고려해 최상의 뉴스제목을 뽑아내는 20년 경력의 뉴스제목교정가이드다.\n\n## Instruction\n교정이 필요한 기사 제목을 입력받아, 맞춤법과 띄어쓰기 오류, 문법 오류를 지적하고 고친 제목을 제시하세요.  \n아래 단계로 진행합니다:  \n1. 입력된 기사 제목을 면밀히 분석하여 맞춤법 오류, 띄어쓰기 실수, 문법 오류 등 문제점을 찾아 지적 항목으로 정리합니다.  \n2. 문제점을 모두 고친 교정된 기사 제목을 결과로 제시합니다.  \n3. 교정이 필요한 부분과 수정결과를 교정이유항목에 작성해주세요.\n4. 기사 제목에 오류가 여러 개 있을 경우, 각 오류를 번호를 매겨 명확히 구분하여 지적합니다.\n5. 독자의 관심을 끌수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.\n6. 어조가 지나치게 감정적이거나 부정적이라면, 적절히 중립적 표현을 사용하세요.\n7. 비속어/욕설등이 포함되어 있다면 이를 제거하고, 의미가 전달될수 있는 적절한 표현으로 수정하세요. \n\n## Output Format\n- 원래제목: [송고한 기사제목]\n- 교정제목: [교정한 기사제목]\n- 교정 이유:\n  1. [교정한 부분과 이유]\n  2. [교정한 부분과 이유]\n\n## Examples\n<예시1>  \n입력: \"코로나19 백신접종율 높히기 위한 대안마련 필요하다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"코로나19 백신 접종률 높이기 위한 대안 마련 시급\"\n- 교정 이유:\n   1. '접종율'은 '접종률'이 맞는 표기입니다.\n   2. '높히기'는 '높이기'로, 맞춤법 오류입니다.\n   3. '대안마련'은 붙여쓰지 않고 '대안 마련'으로 띄어 써야 맞습니다.\n   4. 간결한 어미수정\n\n<예시2>  \n입력: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야한다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야\"\n- 교정 이유:\n  - 간결한 어미 수정\n"
                }
            ]
        },
        # 유저 프롬프트 : 사용자의 실제 입력 데이터
        {
            "role" : "user",
            "content" : [
                {
                    "type" : "text",
                    "text" : "## Input Data\n입력: {피자설기는 유행! 이거는 과연 언제까지?}"
                }
            ]

        }
    ],
    response_format = {"type": "text"}, # 응답 형식
    temperature = 1,                    # 창의성/다양성 (낮으면 결정론적, 일관적 / 높으면 창의적)
    max_completion_tokens = 2048,       # 최대 출력 토큰 
    top_p = 1,                          # 누적확률 p까지의 후보를 샘플링 (1은 전체사용) 
    frequency_penalty = 0,              # 동일 단어 반복시 감점 (반복 억제)
    presence_penalty = 0,               # 이미 등장한 단어는 감점 (반복 억제)
    store = False                       # 응답을 서버에 저장/로깅 할지 여부
)

response

ChatCompletion(id='chatcmpl-EERNQfyRnwgevOlwK8zbDLIEnuFcG', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='- 원래제목: [피자설기는 유행! 이거는 과연 언제까지?]\n- 교정제목: [피자 설기 유행, 과연 언제까지?]\n- 교정 이유:\n  1. ‘피자설기’는 일반적으로 ‘피자 설기’로 띄어 쓰는 것이 자연스럽습니다. 다만 고유 상품명이라면 붙여 쓸 수 있습니다.\n  2. ‘이거는’은 구어적이고 지시 대상이 불분명해 삭제했습니다.\n  3. ‘피자 설기는 유행’의 어색한 문장 구조를 ‘피자 설기 유행’으로 간결하게 다듬어 제목의 주목도를 높였습니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1787110536, model='gpt-5.6-luna', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=342, prompt_tokens=640, total_tokens=982, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=171, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached

In [6]:
print(response.choices[0].message.content)

- 원래제목: [피자설기는 유행! 이거는 과연 언제까지?]
- 교정제목: [피자 설기 유행, 과연 언제까지?]
- 교정 이유:
  1. ‘피자설기’는 일반적으로 ‘피자 설기’로 띄어 쓰는 것이 자연스럽습니다. 다만 고유 상품명이라면 붙여 쓸 수 있습니다.
  2. ‘이거는’은 구어적이고 지시 대상이 불분명해 삭제했습니다.
  3. ‘피자 설기는 유행’의 어색한 문장 구조를 ‘피자 설기 유행’으로 간결하게 다듬어 제목의 주목도를 높였습니다.


In [11]:
    # 기사 제목 교정 API 호출 -> 결과 텍스트만 반환하는 함수
def correct_headline(headline, /, *, model = 'gpt-5.6-luna', temperature = 1, top_p = 1, max_completion_tokens = 2048):
    response = client.chat.completions.create(
        model = 'gpt-5.6-luna',
        messages = [
            # 시스템 프롬프트 : 모델의 페르소나 / 규칙 설정
            {
                "role" : "system",
                "content" : [
                    {
                        "type" : "text",
                        # 모델의 역할 / 출력 예시 / 출력 규칙
                        "text" : "기자들이 송고한 제목에서 맞춤법/문법/의미/어조등을 고려해 최상의 뉴스제목을 뽑아내는 20년 경력의 뉴스제목교정가이드다.\n\n## Instruction\n교정이 필요한 기사 제목을 입력받아, 맞춤법과 띄어쓰기 오류, 문법 오류를 지적하고 고친 제목을 제시하세요.  \n아래 단계로 진행합니다:  \n1. 입력된 기사 제목을 면밀히 분석하여 맞춤법 오류, 띄어쓰기 실수, 문법 오류 등 문제점을 찾아 지적 항목으로 정리합니다.  \n2. 문제점을 모두 고친 교정된 기사 제목을 결과로 제시합니다.  \n3. 교정이 필요한 부분과 수정결과를 교정이유항목에 작성해주세요.\n4. 기사 제목에 오류가 여러 개 있을 경우, 각 오류를 번호를 매겨 명확히 구분하여 지적합니다.\n5. 독자의 관심을 끌수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.\n6. 어조가 지나치게 감정적이거나 부정적이라면, 적절히 중립적 표현을 사용하세요.\n7. 비속어/욕설등이 포함되어 있다면 이를 제거하고, 의미가 전달될수 있는 적절한 표현으로 수정하세요. \n\n## Output Format\n- 원래제목: [송고한 기사제목]\n- 교정제목: [교정한 기사제목]\n- 교정 이유:\n  1. [교정한 부분과 이유]\n  2. [교정한 부분과 이유]\n\n## Examples\n<예시1>  \n입력: \"코로나19 백신접종율 높히기 위한 대안마련 필요하다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"코로나19 백신 접종률 높이기 위한 대안 마련 시급\"\n- 교정 이유:\n   1. '접종율'은 '접종률'이 맞는 표기입니다.\n   2. '높히기'는 '높이기'로, 맞춤법 오류입니다.\n   3. '대안마련'은 붙여쓰지 않고 '대안 마련'으로 띄어 써야 맞습니다.\n   4. 간결한 어미수정\n\n<예시2>  \n입력: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야한다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야\"\n- 교정 이유:\n  - 간결한 어미 수정\n"
                    }
                ]
            },
            # 유저 프롬프트 : 사용자의 실제 입력 데이터
            {
                "role" : "user",
                "content" : [
                    {
                        "type" : "text",
                        "text" : f"## Input Data\n입력: {headline}"
                    }
                ]

            }
        ],
        response_format = {"type": "text"}, # 응답 형식
        temperature = 1,                    # 창의성/다양성 (낮으면 결정론적, 일관적 / 높으면 창의적)
        max_completion_tokens = 2048,       # 최대 출력 토큰 
        top_p = 1,                          # 누적확률 p까지의 후보를 샘플링 (1은 전체사용) 
        frequency_penalty = 0,              # 동일 단어 반복시 감점 (반복 억제)
        presence_penalty = 0,               # 이미 등장한 단어는 감점 (반복 억제)
        store = False                       # 응답을 서버에 저장/로깅 할지 여부
    )

    return response.choices[0].message.content

- 함수 인자 /, *
- / : /의 왼쪽 매개변수는 위치인자방식으로 호출 강제화 (headline은 무조건 첫번째 위치)
-   \* : *의 오른쪽 매개변수는 키워드인자방식으로만 호출 강제화 (model-..., temperature =... 키워드방식으로 사용)

In [12]:
print(correct_headline('피자 설기 유행...언제가지 이어질까'))

- 원래제목: 피자 설기 유행...언제가지 이어질까
- 교정제목: 피자 설기 유행…언제까지 이어질까
- 교정 이유:
  1. ‘언제가지’는 ‘언제까지’로 수정해야 합니다. ‘까지’의 철자를 잘못 쓴 맞춤법 오류입니다.
  2. 말줄임표는 마침표 3개(…)보다 문장부호 ‘…’를 사용하는 것이 적절합니다.


In [13]:
print(correct_headline('피자 설기 열풍, 언제까지 이어질까', model = 'gpt-5.6-sol'))

- 원래제목: 피자 설기 열풍, 언제까지 이어질까
- 교정제목: 피자 설기 열풍, 언제까지 이어질까
- 교정 이유:
  1. 맞춤법과 띄어쓰기상 오류가 없어 원문을 유지했습니다.
  2. ‘언제까지 이어질까’는 열풍의 지속 여부를 묻는 표현으로, 제목의 관심을 유도하면서도 자연스럽습니다.


In [14]:
headlines =[
    "피자 설기 너무 맛있다!",
    "성수동 팝업스토어 사람이 너무 많다 미어터진다",
    "가을 야구 가는팀은?"
]

for headline in headlines:
    output = correct_headline(headline)
    print(output + "\n")

- 원래제목: 피자 설기 너무 맛있다!
- 교정제목: 피자 설기, 이색적인 맛으로 눈길
- 교정 이유:
  1. ‘피자 설기’와 뒤 문장 사이에 쉼표를 넣어 제목의 의미 단위를 명확히 했습니다.
  2. ‘너무 맛있다!’는 주관적이고 감정적인 표현이므로, 뉴스 제목에 맞게 ‘이색적인 맛으로 눈길’처럼 중립적이고 객관적인 표현으로 수정했습니다.

- 원래제목: 성수동 팝업스토어 사람이 너무 많다 미어터진다
- 교정제목: 성수동 팝업스토어, 방문객 몰려 ‘인산인해’

- 교정 이유:
  1. ‘성수동 팝업스토어 사람이’는 조사 ‘에’를 넣어 ‘성수동 팝업스토어에’로 쓰는 것이 자연스럽습니다.
  2. ‘사람’은 기사 제목에 어울리는 ‘방문객’으로 구체화했습니다.
  3. ‘너무 많다’와 ‘미어터진다’는 의미가 중복되고 구어적인 표현이므로, ‘방문객 몰려’와 ‘인산인해’로 간결하고 중립적으로 다듬었습니다.

- 원래제목: 가을 야구 가는팀은?
- 교정제목: 가을 야구에 진출할 팀은?

- 교정 이유:
  1. ‘가는팀’은 ‘가는 팀’으로 띄어 써야 합니다.
  2. ‘가을 야구에 진출할 팀은?’으로 다듬어 구어적인 표현을 줄이고, 기사 제목에 맞게 의미를 명확히 했습니다.



In [18]:
    # 기사 제목 교정 API 호출 -> 결과 텍스트만 반환하는 함수
def chef_json(user_input: str, /, *, model = 'gpt-5.6-luna', temperature = 1, top_p = 1, max_completion_tokens = 2048):
    response = client.chat.completions.create(
        model = 'gpt-5.6-luna',
        messages = [
            # 시스템 프롬프트 : 모델의 페르소나 / 규칙 설정
            {
                "role" : "system",
                "content" : [
                    {
                        "type" : "text",
                        # 모델의 역할 / 출력 예시 / 출력 규칙
                        "text" : "Instruction\n사용자가 입력한 냉장고 내 재료 목록만을 사용하여 만들 수 있는 음식 2가지를 추천하세요.  \n반드시 입력된 재료만 활용하며, 기본양념(간장, 소금, 설탕, 설탕, 식초, 후추 등)은 언제든 사용할 수 있다고 가정하세요.  \n입력된 목록에 없는 재료(양념 제외)는 절대 사용하지 말고, 추가 재료 없이 조리 가능한 음식만 선정하십시오.  \n음식 종류는 반드시 서로 비슷하지 않은 2가지여야 하며, 각 음식에 대한 자세한 요리 레시피(조리 순서)를 단계별로 포함하세요.\n\n- 먼저, 입력 재료로 만들 수 있는 음식 종류를 논리적으로 검토한 뒤, 각 음식이 왜 가능한지 간단히 설명해 주세요.\n- reasoning(논리 및 설명)과 conclusion(최종 추천 결과)은 반드시 JSON의 개별 필드로 구분하여 제공하십시오.\n- reasoning이 반드시 먼저, conclusion이 반드시 마지막에 위치해야 합니다.\n- 결론(conclusion)에는 각 음식명과 단계별 레시피를 포함하세요.\n- 반드시 모든 답변을 한글로 작성하세요.\n\n# Steps\n\n1. 입력 재료만 활용 가능한 음식 2가지를 선정하고, 서로 비슷하지 않은지 확인하세요.\n2. reasoning(논리/설명) 필드에: \n    - 해당 재료로 어떤 음식이 가능한지, 그 이유를 간단히 단계별 논리로 설명하세요.\n3. conclusion(최종 추천) 필드에:\n    - 각 음식의 음식명\n    - 해당 음식의 구체적 요리 레시피(순서대로 단계를 나열, 최소 3단계 이상)\n    - 위 구조로 2가지만 반드시 작성하세요.\n\n# Output Format\n\n모든 답변은 아래 JSON 구조로 출력하세요.  \n- \"reasoning\": 각 음식이 왜 가능한지 단계별 논리와 검토(한글 서술, 리스트)\n- \"conclusion\": 음식명과 상세한 단계별 레시피(한글 서술, 리스트. 각 요소는 {\"food_name\": \"음식명\", \"recipe: [\"레시피1\", \"레시피2\"]} 형식)\n\n# Examples\n\n사용자 입력 예시:\n- 입력: 계란, 양파, 당근\n\n출력 예시(JSON):\n\n{\n  \"reasoning\": [\n    \"계란, 양파, 당근만 사용하여 만들 수 있는 요리를 검토합니다.\",\n    \"계란과 채소(양파, 당근)만으로 달걀전이 가능합니다. 채소를 잘게 썰어 계란과 섞어 부치면 완성할 수 있습니다.\",\n    \"계란찜 역시 이 재료로 만들 수 있습니다. 계란을 풀고 다진 채소를 섞은 후, 찜기를 사용해 익히면 완성됩니다.\"\n  ],\n  \"conclusion\": [\n    {\n      \"food_name\": \"달걀전\",\n      \"recipe\": [\n        \"1. 양파와 당근을 잘게 썰어줍니다.\",\n        \"2. 계란을 풀고 썰어둔 양파와 당근, 소금, 후추를 넣고 섞습니다.\",\n        \"3. 달궈진 팬에 기름을 두르고 반죽을 얇게 올린 후, 앞뒤로 노릇하게 부칩니다.\"\n      ]\n    },\n    {\n      \"food_name\": \"계란찜\",\n      \"recipe\": [\n        \"1. 계란을 볼에 넣고 곱게 풀어줍니다.\",\n        \"2. 다진 양파와 당근, 소금, 후추를 계란물에 넣고 섞습니다.\",\n        \"3. 뚝배기나 내열 용기에 재료를 옮겨 담고, 중탕 또는 전자레인지로 익혀 부드럽게 완성합니다.\"\n      ]\n    }\n  ]\n}\n\n(실제 예시는 입력 재료와 음식에 따라 달라지며, 각 음식의 레시피 단계는 3단계 이상, 충분히 구체적으로 작성하십시오.)\n\n# Notes\n\n- 반드시 입력 재료만 사용하고, 음식명 및 조리법 전부 한글로 기입하세요.\n- 각 추천 요리는 서로 다른 종류여야 하며, 각 음식마다 레시피 단계는 구체적이고 논리적으로 작성돼야 합니다.\n- reasoning(논리/설명) → conclusion(최종 추천 및 레시피) 순서는 꼭 지켜야 합니다.\n- 답변 형식은 반드시 JSON이어야 하며, 한글로만 작성하세요.\n\n[중요: 2가지 음식 추천, 상세 단계별 한글 레시피, 입력 재료만 허용, 양념장은 보유 가정, 항상 reasoning이 먼저, conclusion이 뒤, 반드시 JSON, 예시 구조 참고, 모든 답변은 한글로!]"
                    },
                ]
            },
            # 유저 프롬프트 : 사용자의 실제 입력 데이터
            {
                "role" : "user",
                "content" : [
                    {
                        "type" : "text",
                        "text" : user_input
                    }
                ]

            }
        ],
        response_format = {"type": "text"}, # 응답 형식
        temperature = 1,                    # 창의성/다양성 (낮으면 결정론적, 일관적 / 높으면 창의적)
        max_completion_tokens = 2048,       # 최대 출력 토큰 
        top_p = 1,                          # 누적확률 p까지의 후보를 샘플링 (1은 전체사용) 
        frequency_penalty = 0,              # 동일 단어 반복시 감점 (반복 억제)
        presence_penalty = 0,               # 이미 등장한 단어는 감점 (반복 억제)
        store = False                       # 응답을 서버에 저장/로깅 할지 여부
    )

    return json.loads(response.choices[0].message.content) # json -> python dict 파싱

In [19]:
chef_json("계란, 양파, 삼겹살")

{'reasoning': ['입력된 재료는 계란, 양파, 삼겹살이며, 간장·소금·설탕·식초·후추 같은 기본양념은 사용할 수 있다고 가정합니다.',
  '삼겹살은 자체 지방이 있어 별도의 식용유 없이 구울 수 있고, 양파와 함께 볶으면 삼겹살 양파볶음을 만들 수 있습니다.',
  '계란은 물과 소금으로 익히면 찜 요리가 가능하며, 익힌 삼겹살과 양파를 넣어 삼겹살 계란찜을 만들 수 있습니다.',
  '첫 번째 음식은 팬에 볶는 요리이고 두 번째 음식은 부드럽게 쪄내는 요리이므로 조리법과 식감이 서로 다릅니다.'],
 'conclusion': [{'food_name': '삼겹살 양파볶음',
   'recipe': ['1. 삼겹살을 먹기 좋은 크기로 자르고, 양파는 굵게 채 썹니다.',
    '2. 달군 팬에 삼겹살을 넣고 중간 불에서 뒤집어가며 충분히 익힙니다. 삼겹살에서 지방이 나오므로 별도의 기름은 넣지 않습니다.',
    '3. 삼겹살이 노릇하게 익으면 팬에 남은 기름을 취향에 따라 조금 덜어냅니다.',
    '4. 채 썬 양파를 넣고 삼겹살과 함께 볶아 양파가 투명하고 부드러워질 때까지 익힙니다.',
    '5. 간장, 설탕, 후추를 넣고 재료에 양념이 고르게 배도록 1~2분 더 볶아 완성합니다.']},
  {'food_name': '삼겹살 계란찜',
   'recipe': ['1. 삼겹살을 잘게 썰어 팬에 볶고, 속까지 완전히 익힌 뒤 잠시 둡니다.',
    '2. 양파를 잘게 다져 익힌 삼겹살과 섞습니다.',
    '3. 계란을 그릇에 깨 넣고 물과 소금을 넣어 충분히 풀어줍니다.',
    '4. 계란물에 볶은 삼겹살과 다진 양파를 넣고 고르게 섞습니다.',
    '5. 혼합물을 내열 용기나 냄비에 붓고 약한 불에서 뚜껑을 덮어 천천히 익힙니다.',
    '6. 계란물이 대부분 굳고 가운데까지 익으면 불을 끄고 잠시 뜸을 들인 뒤 후추를 뿌려 완성합니다.']}]}

In [22]:
output = chef_json("계란, 양파, 삼겹살, 찬밥")

for food in output['conclusion']:
    print(f"추천 음식 : {food['food_name']}")
    print('레시피 : ')
    for step in food['recipe']:
        print(' ', step)
    print()

추천 음식 : 삼겹살 양파구이
레시피 : 
  1. 삼겹살을 먹기 좋은 크기로 자르고, 양파는 굵게 채 썹니다.
  2. 달군 팬에 삼겹살을 올려 중간 불에서 굽습니다. 삼겹살에서 기름이 나오면 고기를 뒤집어 양면을 익힙니다.
  3. 삼겹살이 거의 익으면 팬 한쪽에 양파를 넣고 삼겹살 기름에 함께 볶습니다.
  4. 양파가 투명해지고 삼겹살이 노릇하게 익을 때까지 볶은 뒤 소금과 후추로 간합니다.
  5. 삼겹살과 양파를 고루 섞어 한 번 더 볶고, 고기가 완전히 익었는지 확인한 후 담아냅니다.

추천 음식 : 삼겹살 계란볶음밥
레시피 : 
  1. 삼겹살을 잘게 자르고, 양파도 작게 다집니다. 찬밥은 덩어리를 미리 풀어둡니다.
  2. 팬에 잘게 썬 삼겹살을 넣고 중간 불에서 볶아 기름을 충분히 냅니다.
  3. 삼겹살이 익으면 양파를 넣고 양파가 투명해질 때까지 볶습니다.
  4. 볶은 재료를 팬 한쪽으로 밀고 빈 공간에 계란을 깨 넣어 저으면서 익힙니다.
  5. 계란이 반쯤 익으면 찬밥을 넣고 주걱으로 덩어리를 풀면서 삼겹살, 양파, 계란과 골고루 섞어 볶습니다.
  6. 소금, 후추, 간장 등 기본양념으로 간하고 밥알이 고슬고슬해질 때까지 볶아 완성합니다.



In [1]:
import json
from openai import OpenAI

client = OpenAI()


def job_interview_json(
    user_input: str,
    /,
    *,
    model: str = "gpt-5.6-luna",
    temperature: float = 1,
    top_p: float = 1,
    max_completion_tokens: int = 2048,
) -> dict:
    system_prompt = """
# Instruction

당신은 20년 경력의 ML/DL 엔지니어이고, 이번 신입 개발자 채용의 면접관입니다.
주어진 job posting(회사 정보)을 바탕으로 신입 개발자에게 제공할 면접 질문과
모범답안을 작성하세요.

다음 지침을 반드시 준수하세요.

- 모든 질문과 답변은 한글로 작성합니다.
- 지원자가 신입 개발자임을 반영합니다.
- hard skill과 soft skill/leadership 영역을 구분합니다.
- 각 영역에 question과 answer 쌍을 최소 2개 작성합니다.
- 전체 질문은 최소 4개 이상 작성합니다.
- 입력 정보가 부족하면 일반적인 산업 및 포지션을 기준으로 추론합니다.
- 입력값이 없다면 [회사정보], [스펙] 등의 placeholder를 사용합니다.
- 유효한 JSON 객체만 출력합니다.
- JSON 이외의 설명이나 마크다운 코드 블록은 출력하지 않습니다.

# Output Format

{
  "hard_skill": [
    {
      "question": "[기술 중심 면접 질문]",
      "answer": "[신입 개발자 입장의 모범답안]"
    }
  ],
  "soft_skill_leadership": [
    {
      "question": "[태도, 소통, 리더십 및 성장 가능성 중심 질문]",
      "answer": "[신입 개발자 입장의 모범답안]"
    }
  ]
}
""".strip()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_input,
            },
        ],
        response_format={"type": "json_object"},
        temperature=temperature,
        top_p=top_p,
        max_completion_tokens=max_completion_tokens,
        frequency_penalty=0,
        presence_penalty=0,
        store=False,
    )

    content = response.choices[0].message.content

    if not content:
        raise ValueError("모델이 빈 응답을 반환했습니다.")

    output = json.loads(content)

    required_keys = {"hard_skill", "soft_skill_leadership"}
    missing_keys = required_keys - output.keys()

    if missing_keys:
        raise ValueError(
            f"응답 JSON에 필수 키가 없습니다: {sorted(missing_keys)}"
        )

    return output

In [2]:
user_input = """
회사정보:
- 제조공정 데이터를 분석하는 AI 솔루션 기업
- Python 기반 머신러닝 모델 및 데이터 파이프라인 개발
- SQL을 이용한 데이터 추출 및 분석 업무

지원자 스펙:
- 컴퓨터공학 전공
- 신입 개발자
- Python, SQL 사용 가능
- 머신러닝 프로젝트 경험 있음
"""

output = job_interview_json(user_input)

In [3]:
for category, qa_list in output.items():
    print(f"[{category}]")

    for qa in qa_list:
        question = qa.get("question", "").strip()
        answer = qa.get("answer", "").strip()

        print(f"Q : {question}")
        print(f"A : {answer}")
        print()

    print()

[hard_skill]
Q : 제조공정 데이터로 불량품 예측 모델을 개발한다면, 일반적으로 어떤 순서로 진행하시겠습니까?
A : 먼저 문제의 목표와 평가 지표를 명확히 정의하겠습니다. 이후 SQL이나 Python을 활용해 데이터를 추출하고, 결측치·이상치·중복 데이터를 확인하겠습니다. 제조공정의 시간 순서가 중요할 수 있으므로 데이터 누수 여부를 점검한 뒤 학습·검증·테스트 데이터를 적절히 분리하겠습니다. 그다음 기준 모델을 만든 후 특성 전처리와 모델을 개선하고, 정확도뿐 아니라 정밀도·재현율·F1 점수와 같은 지표를 비교하겠습니다. 마지막으로 실제 공정에서 사용할 수 있도록 예측 결과를 해석하고 모델의 성능을 지속적으로 모니터링하겠습니다.

Q : Python으로 머신러닝 데이터 파이프라인을 개발할 때 중요하게 고려할 사항은 무엇인가요?
A : 재현성과 안정성을 중요하게 생각합니다. 먼저 데이터 로딩, 전처리, 학습, 평가 단계를 함수나 모듈로 분리하고, Pandas와 Scikit-learn을 활용해 일관된 처리 흐름을 구성하겠습니다. 전처리 기준이나 모델 파라미터는 설정값으로 관리하고, 예외 처리와 로그를 추가해 문제가 발생한 지점을 쉽게 확인할 수 있도록 하겠습니다. 또한 학습 데이터와 운영 데이터의 스키마가 다르지 않은지 검증하고, Git을 사용해 코드 변경 이력을 관리하겠습니다. 신입이므로 처음에는 단순하고 명확한 구조로 구현한 뒤 리뷰를 통해 개선하겠습니다.

Q : SQL로 제조공정 데이터에서 설비별 평균 불량률과 생산량을 조회해야 한다면 어떻게 작성하시겠습니까?
A : 설비 식별자와 생산 결과가 있는 테이블을 기준으로 설비별로 그룹화하겠습니다. 예를 들어 생산 수량과 불량 수량이 있다면 SUM으로 각각 집계하고, 불량률은 불량 수량을 생산 수량으로 나누어 계산할 수 있습니다. 생산 수량이 0인 경우에는 NULLIF를 사용해 0으로 나누는 오류를 방지하겠습니다. 개념적으로는 다음과 같은 형태입니다: SELECT 설비ID, SUM(생산수량) AS 

In [4]:
output

{'hard_skill': [{'question': '제조공정 데이터로 불량품 예측 모델을 개발한다면, 일반적으로 어떤 순서로 진행하시겠습니까?',
   'answer': '먼저 문제의 목표와 평가 지표를 명확히 정의하겠습니다. 이후 SQL이나 Python을 활용해 데이터를 추출하고, 결측치·이상치·중복 데이터를 확인하겠습니다. 제조공정의 시간 순서가 중요할 수 있으므로 데이터 누수 여부를 점검한 뒤 학습·검증·테스트 데이터를 적절히 분리하겠습니다. 그다음 기준 모델을 만든 후 특성 전처리와 모델을 개선하고, 정확도뿐 아니라 정밀도·재현율·F1 점수와 같은 지표를 비교하겠습니다. 마지막으로 실제 공정에서 사용할 수 있도록 예측 결과를 해석하고 모델의 성능을 지속적으로 모니터링하겠습니다.'},
  {'question': 'Python으로 머신러닝 데이터 파이프라인을 개발할 때 중요하게 고려할 사항은 무엇인가요?',
   'answer': '재현성과 안정성을 중요하게 생각합니다. 먼저 데이터 로딩, 전처리, 학습, 평가 단계를 함수나 모듈로 분리하고, Pandas와 Scikit-learn을 활용해 일관된 처리 흐름을 구성하겠습니다. 전처리 기준이나 모델 파라미터는 설정값으로 관리하고, 예외 처리와 로그를 추가해 문제가 발생한 지점을 쉽게 확인할 수 있도록 하겠습니다. 또한 학습 데이터와 운영 데이터의 스키마가 다르지 않은지 검증하고, Git을 사용해 코드 변경 이력을 관리하겠습니다. 신입이므로 처음에는 단순하고 명확한 구조로 구현한 뒤 리뷰를 통해 개선하겠습니다.'},
  {'question': 'SQL로 제조공정 데이터에서 설비별 평균 불량률과 생산량을 조회해야 한다면 어떻게 작성하시겠습니까?',
   'answer': '설비 식별자와 생산 결과가 있는 테이블을 기준으로 설비별로 그룹화하겠습니다. 예를 들어 생산 수량과 불량 수량이 있다면 SUM으로 각각 집계하고, 불량률은 불량 수량을 생산 수량으로 나누어 계산할 수 있습니다. 생산 수량이 0인 경우에는 NUL